# Titanic Survival Prediction
Examples of inference using 3 architectural patterns:

1. Scikit-Learn -> Native Pandas: Single-node driver inference (low latency for small datasets)
2. Scikit-Learn -> Spark UDF: Distributed cluster interence via Apache Arrow (scales single-node packages to billions of rows)
3. SparkML -> Native Spark Pipeline: High-performance JVM-native execution using Spark's `.transform()` engine


In [0]:
import os
import mlflow
import pandas as pd

from pyspark.sql.functions import col, struct

In [0]:
SKLEARN_MODEL_URI = "models:/titanic_sklearn_model/1"
SPARKML_MODEL_URI = "models:/titanic_sparkml_model/1"

TITANIC_FEATURES = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
LOCAL_TEST_CSV = "test.csv"


In [0]:
# 1. Load the model as a generic Python function from MLflow into driver memory
pandas_model = mlflow.pyfunc.load_model(SKLEARN_MODEL_URI)

# 2. Read the local repo CSV into a standard Pandas DataFrame
pd_df = pd.read_csv(LOCAL_TEST_CSV)

# 3. Quick feature preprocessing (Filling NaNs to match typical training workflows)
pd_df['Age'] = pd_df['Age'].fillna(pd_df['Age'].median())
pd_df['Fare'] = pd_df['Fare'].fillna(pd_df['Fare'].median())

# 4. Extract features, execute prediction, and format like gender_submission.csv
X_pandas = pd_df[TITANIC_FEATURES]
pd_df['Survived'] = pandas_model.predict(X_pandas)

pandas_submission_df = pd_df[['PassengerId', 'Survived']]

print("Option 1 (Pandas Native) Output Preview:")
display(pandas_submission_df.head(5))

In [0]:
# 1. Read the local CSV directly into a distributed Spark DataFrame
absolute_csv_path = os.path.abspath(LOCAL_TEST_CSV)
raw_spark_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"file:{absolute_csv_path}")

# Fill nulls and cast data types to ensure alignment with the UDF execution environment
spark_df_cleaned = raw_spark_df.na.fill({"Age": 29.69, "Fare": 32.20}) \
                               .withColumn("Age", col("Age").cast("double")) \
                               .withColumn("Fare", col("Fare").cast("double"))

# 2. Turn the MLflow model into a distributed Spark User Defined Function (UDF)
spark_predict_udf = mlflow.pyfunc.spark_udf(spark, model_uri=SKLEARN_MODEL_URI)

# 3. Apply the UDF across the cluster by packaging features inside a PySpark Struct
spark_predictions_df = spark_df_cleaned.withColumn(
    "Survived", 
    spark_predict_udf(struct(*[col(c) for c in TITANIC_FEATURES]))
)

# 4. Isolate columns to match the gender_submission layout
spark_submission_df = spark_predictions_df.select("PassengerId", "Survived")

print("Option 2 (Spark UDF Distributed) Output Preview:")
display(spark_submission_df.limit(5))

In [0]:
# 1. Load the native SparkML PipelineModel out of MLflow
# (This preserves internal feature transformers like VectorAssembler logged during training)
native_sparkml_pipeline = mlflow.spark.load_model(SPARKML_MODEL_URI)

# 2. Execute distributed scoring natively using Spark's .transform() API
# (Data stays entirely inside the JVM engine—zero conversion overhead to Python)
sparkml_output_df = native_sparkml_pipeline.transform(spark_df_cleaned)

# 3. Extract predictions (SparkML defaults prediction column outputs to floats, cast to Int)
sparkml_submission_df = sparkml_output_df.select(
    col("PassengerId"),
    col("prediction").cast("int").alias("Survived")
)

print("Option 3 (SparkML Native Transform) Output Preview:")
display(sparkml_submission_df.limit(5))